# 3. Logistic regression: L1, L2 and Elastic Net

**Question: does the *form* of shrinkage change the maintenance cost, or only the
coefficient count?**

Logistic regression is often included as a token baseline and left untuned. That wastes
it. With 171 anonymised features of unknown redundancy, the choice between an L1 penalty
(which zeroes coefficients outright), an L2 penalty (which shrinks them together) and
Elastic Net (which does both) is a genuine modelling question — and it interacts with
class weighting, because upweighting the rare class changes which coefficients survive.

Candidates are sampled over penalty, `C`, `l1_ratio` and positive-class weight, then
scored on **tuning-set cost**, never on accuracy.

> **The objective.** Every number in this notebook is judged against
> `J = 10·FP + 500·FN`. A false positive is an unnecessary inspection; a false
> negative is a truck that fails in service. Missing one failure costs as much
> as fifty needless inspections, and that ratio is what makes the modelling
> choices here matter.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv
from scania_aps.plotting import apply_house_style

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"

apply_house_style()

train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)

pd.DataFrame(
    {
        "trucks": [len(train.y), len(test.y)],
        "features": [train.X.shape[1], test.X.shape[1]],
        "failures": [int(train.y.sum()), int(test.y.sum())],
        "failure_rate": [train.y.mean(), test.y.mean()],
    },
    index=["training set", "official test set"],
)

## Search

Each candidate is fitted on the `fit` subset and scored on the `tune` subset. For every
candidate the threshold is re-optimised, so configurations are compared at their own
best operating point rather than at an arbitrary shared one.

In [ ]:
from scania_aps.optimization import tune_logistic
from scania_aps.split import development_split

split = development_split(train.X, train.y)
best, trace = tune_logistic(
    split.X_fit, split.y_fit, split.X_tune, split.y_tune, n_trials=36
)

results = pd.DataFrame(
    [
        {
            "penalty": r.config.penalty,
            "C": r.config.C,
            "l1_ratio": r.config.l1_ratio,
            "class_weight": r.config.positive_class_weight,
            "threshold": r.threshold,
            "tune_cost": r.tune_cost,
            "pr_auc": r.pr_auc,
        }
        for r in trace
    ]
).sort_values("tune_cost")

print(f"best configuration: {best}")
results.head(12)

## Cost against regularization strength

`C` is inverse regularization: small `C` means heavy shrinkage. Each penalty family is
its own series, so the reader can see whether the families genuinely separate or whether
the spread is dominated by `C` alone. The minimum of each series is marked.

In [ ]:
from scania_aps.plotting import series_lines

# Bin C on a log grid so the three penalties are comparable at the same strengths.
grid = np.logspace(np.log10(results["C"].min()), np.log10(results["C"].max()), 7)
results["C_bin"] = pd.cut(results["C"], bins=grid, include_lowest=True)
centres = np.sqrt(grid[:-1] * grid[1:])

series = {}
for penalty in ["l1", "l2", "elasticnet"]:
    subset = results[results["penalty"] == penalty]
    if subset.empty:
        continue
    binned = subset.groupby("C_bin", observed=False)["tune_cost"].min()
    series[penalty] = binned.to_numpy(dtype=float)

fig, ax = series_lines(
    centres,
    series,
    title="Cost against regularization strength, by penalty",
    subtitle="Best tuning cost within each C band. Lower is better; marked point is each family's minimum.",
    xlabel="C  (inverse regularization strength, log scale)",
    ylabel="Tuning-set cost",
    log_x=True,
    mark_minimum=True,
)

## What survives the L1 penalty?

If the winning configuration uses L1 or Elastic Net, some coefficients are exactly zero.
The count matters: a model that keeps 30 of 171 features is telling you the sensor set is
highly redundant, which is worth knowing independently of the cost.

Note the `missing::` prefix — those are the missingness indicators from
[notebook 01](01_data_quality_and_missingness.ipynb). If they survive L1 selection, the
absence of a reading is carrying real predictive weight.

In [ ]:
from scania_aps.feature_selection import l1_nonzero_features
from scania_aps.models.logistic import build_logistic_pipeline
from scania_aps.plotting import magnitude_bars

model = build_logistic_pipeline(best).fit(split.X_fit, split.y_fit)

if best.penalty in {"l1", "elasticnet"}:
    kept = l1_nonzero_features(model)
    print(f"non-zero coefficients: {len(kept)}")
    indicators = [f for f in kept if f.feature.startswith("missing::")]
    print(f"of which missingness indicators: {len(indicators)}")

    top = kept[:18]
    fig, ax = magnitude_bars(
        [f.feature for f in top],
        [abs(f.score) for f in top],
        title="Largest surviving coefficients",
        subtitle=f"{best.penalty} penalty, C={best.C:.4g}. Bars show |coefficient|.",
        xlabel="|coefficient|",
        value_format="{:.3f}",
    )
    display(pd.DataFrame([f.__dict__ for f in kept]).head(25))
else:
    print(f"The winning penalty was {best.penalty}, which does not produce exact zeros.")
    print("Coefficient sparsity is not defined for this configuration.")

### Reading the result

Compare the three series in the chart above. If they overlap heavily, the penalty *form*
is not what matters here and the honest conclusion is that regularization strength
dominates. If L1 sits consistently below the others, the feature set is redundant enough
that discarding features outright helps.

Either way the conclusion is stated in cost, not in accuracy or AUC.

**Next:** [04 — linear SVM](04_svm_margin_regularization.ipynb), which asks the same
question of a model that emits margins rather than probabilities.